In [1]:
import pandas as pd
import numpy as np


# Đọc dữ liệu từ file csv

In [2]:
file_path = "../../DATA EXPLORER CONTEST/Data - 12ndMar/6.2 (his) financialreport_metrics_FPT_CMG_processed.csv"

dataframe = pd.read_csv(file_path)

dataframe.head()

,Indicator,StockID,Q1_2023,Q1_2024,Q2_2023,Q2_2024,Q3_2023,Q3_2024,Q4_2023,Q4_2024
0,Beta\nLần,CMG,1.02,0.93,1.04,1.18,0.94,1.19,1.01,NaN
1,Beta\nLần,FPT,0.78,0.76,0.74,0.80,0.75,0.81,0.77,0.96
2,Chi phí bán hàng/Doanh thu thuần\n%,CMG,5.61,6.70,6.34,6.01,5.75,5.82,8.53,NaN
3,Chi phí bán hàng/Doanh thu thuần\n%,FPT,9.43,9.71,9.92,10.42,10.42,9.83,8.96,9.57
4,Chi phí lãi vay/Doanh thu thuần\n%,CMG,1.16,1.02,1.27,0.93,1.17,1.05,0.97,NaN


# Lọc ra các hàng cần thiết

In [3]:
# Lọc theo nhiều điều kiện Indicator và StockID
filtered_df = dataframe[
    (
        (dataframe['Indicator'].str.contains("EPS", regex=True)) |
        (dataframe['Indicator'].str.contains("P/E", regex=True)) |
        (dataframe['Indicator'].str.contains("P/B", regex=True)) |
        (dataframe['Indicator'].str.contains("P/S", regex=True)) |
        (dataframe['Indicator'].str.contains("Beta", regex=True))|
        (dataframe['Indicator'].str.contains("ROE", regex=True)) |
        (dataframe['Indicator'].str.contains("ROA", regex=True)) |
        (dataframe['Indicator'].str.contains("ROC", regex=True)) 
    ) &
    (dataframe['StockID'] == 'FPT')
]
filtered_df

,Indicator,StockID,Q1_2023,Q1_2024,Q2_2023,Q2_2024,Q3_2023,Q3_2024,Q4_2023,Q4_2024
1,Beta\nLần,FPT,0.78,0.76,0.74,0.80,0.75,0.81,0.77,0.96
9,Chỉ số giá thị trường trên doanh thu thuần (P/...,FPT,7.43,10.50,7.61,12.50,8.56,12.35,8.31,12.74
11,Chỉ số giá thị trường trên giá trị sổ sách (P/...,FPT,3.21,4.66,3.32,5.81,4.15,5.55,4.08,6.27
13,Chỉ số giá thị trường trên thu nhập (P/E)\nLần,FPT,15.04,21.02,16.26,23.23,17.31,23.80,17.55,26.77
41,Thu nhập trên mỗi cổ phần của 4 quý gần nhất (...,FPT,"5,258","5,541","5,289","5,618","5,362","5,652","5,476","5,697"
90,Tỷ suất lợi nhuận trên vốn chủ sở hữu bình quâ...,FPT,5.70,5.83,5.42,5.81,6.10,6.13,5.92,5.87
94,Tỷ suất sinh lợi trên tổng tài sản bình quân (...,FPT,2.92,2.94,2.71,2.94,2.84,3.14,2.82,2.99
96,Tỷ suất sinh lợi trên vốn dài hạn bình quân (R...,FPT,8.28,8.39,8.54,8.40,9.25,8.61,8.85,8.38


# Điền dữ liệu vào bảng

In [4]:
# Tạo một danh sách các ngày từ 26/3/2024 đến 12/3/2025
date_range = pd.date_range(start='2024-03-26', end='2025-03-12')

def process_bang_chi_so(filtered_df, start_date, end_date):
    # Tạo danh sách ngày và khởi tạo bảng chỉ số
    date_range = pd.date_range(start=start_date, end=end_date)
    bang_chi_so = pd.DataFrame({'Date': date_range})
    bang_chi_so['Beta'] = np.nan
    bang_chi_so['P/S'] = np.nan
    bang_chi_so['P/B'] = np.nan
    bang_chi_so['P/E'] = np.nan
    bang_chi_so['EPS'] = np.nan
    bang_chi_so['ROE'] = np.nan
    bang_chi_so['ROA'] = np.nan
    bang_chi_so['ROC'] = np.nan
    # Định nghĩa các điều kiện theo quý
    quarter_conditions = {
        'Q4_2023': lambda date: 1 <= date.month <= 3 and date.year == 2024,
        'Q1_2024': lambda date: 4 <= date.month <= 6 and date.year == 2024,
        'Q2_2024': lambda date: 7 <= date.month <= 9 and date.year == 2024,
        'Q3_2024': lambda date: 10 <= date.month <= 12 and date.year == 2024,
        'Q4_2024': lambda date: 1 <= date.month <= 3 and date.year == 2025,
    }
    # Lặp qua các cột trong filtered_df
    for item in filtered_df.columns:
        if item in quarter_conditions:  # Kiểm tra nếu cột thuộc các quý
            condition = quarter_conditions[item]
            for i, date in enumerate(bang_chi_so['Date']):
                if condition(date):  # Kiểm tra điều kiện ngày tháng
                    bang_chi_so.loc[i, 'Beta'] = filtered_df[item].values[0]
                    bang_chi_so.loc[i, 'P/S'] = filtered_df[item].values[1]
                    bang_chi_so.loc[i, 'P/B'] = filtered_df[item].values[2]
                    bang_chi_so.loc[i, 'P/E'] = filtered_df[item].values[3]
                    bang_chi_so.loc[i, 'EPS'] = filtered_df[item].values[4]
                    bang_chi_so.loc[i, 'ROE'] = filtered_df[item].values[5]
                    bang_chi_so.loc[i, 'ROA'] = filtered_df[item].values[6]
                    bang_chi_so.loc[i, 'ROC'] = filtered_df[item].values[7]

    return bang_chi_so

bang_chi_so = process_bang_chi_so(filtered_df, '2024-03-26', '2025-03-12')


C:\Users\Dell\AppData\Local\Temp\ipykernel_4912\440307027.py:30: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '0.76' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  bang_chi_so.loc[i, 'Beta'] = filtered_df[item].values[0]
C:\Users\Dell\AppData\Local\Temp\ipykernel_4912\440307027.py:31: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '10.50' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  bang_chi_so.loc[i, 'P/S'] = filtered_df[item].values[1]
C:\Users\Dell\AppData\Local\Temp\ipykernel_4912\440307027.py:32: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '4.66' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  bang_chi_so.l

# Kiểm tra kết quả và lưu file csv

In [5]:
# kiểm tra kết quả
bang_chi_so.head()

,Date,Beta,P/S,P/B,P/E,EPS,ROE,ROA,ROC
0,2024-03-26,0.77,8.31,4.08,17.55,"5,476",5.92,2.82,8.85
1,2024-03-27,0.77,8.31,4.08,17.55,"5,476",5.92,2.82,8.85
2,2024-03-28,0.77,8.31,4.08,17.55,"5,476",5.92,2.82,8.85
3,2024-03-29,0.77,8.31,4.08,17.55,"5,476",5.92,2.82,8.85
4,2024-03-30,0.77,8.31,4.08,17.55,"5,476",5.92,2.82,8.85


In [14]:
# Xuất kết quả ra file CSV
bang_chi_so.to_csv('../../DATA EXPLORER CONTEST/Preprocessed_Data/bang_chi_so_FPT.csv', index=False)